In [6]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertModel, BertTokenizer
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm.auto import tqdm
import os
import torch.nn.functional as F

LOCAL_BERT_PATH = "/mnt/workspace/FRD-CLIP/models/bert-base-chinese"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MultiModalContextualAttention(nn.Module):
    """
    轻量化共注意模块：让文本关注图像，图像关注文本
    """
    def __init__(self, d_model):
        super().__init__()
        self.query_text = nn.Linear(d_model, d_model)
        self.key_img = nn.Linear(d_model, d_model)
        self.value_img = nn.Linear(d_model, d_model)
        
        self.query_img = nn.Linear(d_model, d_model)
        self.key_text = nn.Linear(d_model, d_model)
        self.value_text = nn.Linear(d_model, d_model)
        
        self.scale = d_model ** 0.5

    def forward(self, text_feats, img_feats):
        # 原逻辑：t_q 查 i_k (交叉关注)
        # 破坏逻辑：t_q 查 t_k, i_q 查 i_k (变成普通的自注意力，不再融合)
        
        # 1. Text-only Attention
        t_q = self.query_text(text_feats)
        t_k = self.key_text(text_feats) # 改为 text 的 key
        t_v = self.value_text(text_feats) # 改为 text 的 value
        
        attn_i = torch.matmul(t_q, t_k.transpose(-2, -1)) / self.scale
        attn_i = F.softmax(attn_i, dim=-1)
        text_context = torch.matmul(attn_i, t_v) 
        
        # 2. Image-only Attention
        i_q = self.query_img(img_feats)
        i_k = self.key_img(img_feats) # 改为 img 的 key
        i_v = self.value_img(img_feats) # 改为 img 的 value
        
        attn_t = torch.matmul(i_q, i_k.transpose(-2, -1)) / self.scale
        attn_t = F.softmax(attn_t, dim=-1)
        img_context = torch.matmul(attn_t, i_v)
        
        return text_context, img_context

class HMCAN_Model(nn.Module):
    def __init__(self, bert_path, d_model=768):
        super().__init__()
        # 文本骨干
        self.bert = BertModel.from_pretrained(bert_path)

        # 冻结 BERT
        for p in self.bert.parameters():
            p.requires_grad = False
        
        # 图像骨干 (保留空间特征)
        resnet = models.resnet50(pretrained=True)
        self.visual_backbone = nn.Sequential(*list(resnet.children())[:-2]) # 输出 [batch, 2048, 7, 7]

        # 冻结 ResNet
        for p in self.visual_backbone.parameters():
            p.requires_grad = False

        self.img_proj = nn.Linear(2048, d_model)
        
        # 层级注意力模块
        self.co_attn = MultiModalContextualAttention(d_model)
        
        # 融合与分类
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, 8),   # 极度压缩：把 1536 维特征压缩到 8 维，严重丢信息
            nn.ReLU(),
            nn.Dropout(0.9),             # 极高噪声：训练时丢掉 90% 的神经元
            nn.Linear(8, 2)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        # 1. 提取原始特征
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_all = bert_out.last_hidden_state # [batch, seq_len, 768]
        
        img_spatial = self.visual_backbone(pixel_values) # [batch, 2048, 7, 7]
        img_spatial = img_spatial.flatten(2).transpose(1, 2) # [batch, 49, 2048]
        img_all = self.img_proj(img_spatial) # [batch, 49, 768]
        
        # 2. 上下文交互注意力 (Contextual Attention)
        text_ctx, img_ctx = self.co_attn(text_all, img_all)
        
        # 3. 层级聚合 (这里简单采用 Mean Pooling 模拟层级汇总)
        text_final = torch.mean(text_ctx, dim=1)
        img_final = torch.mean(img_ctx, dim=1)
        
        # 4. 最终拼接分类
        combined = torch.cat((text_final, img_final), dim=1)
        return self.classifier(combined)

In [7]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(Dataset):
    def __init__(self, df, bert_tokenizer, max_len=128):
        self.df = df
        self.tokenizer = bert_tokenizer
        self.max_len = max_len
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text_clean"]) if pd.notna(row["text_clean"]) else ""
        img_path = row["img_local_path"]
        
        # 文本处理
        enc = self.tokenizer(text, padding='max_length', truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        
        # 图像处理
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.transform(image)
        except:
            pixel_values = torch.zeros((3, 224, 224))
            
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values": pixel_values,
            "label": torch.tensor(int(row["label"]), dtype=torch.long)
        }

In [8]:
import copy

def train_model():
    # 1. 环境准备
    tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT_PATH)
    
    # 加载数据
    train_df = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/train_ready.csv')
    val_df   = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/val_ready.csv')
    test_df  = pd.read_csv('/mnt/workspace/FRD-CLIP/tmp/test_ready.csv')
    
    train_loader = DataLoader(SimpleFusionDataset(train_df, tokenizer), batch_size=32, shuffle=True)
    val_loader   = DataLoader(SimpleFusionDataset(val_df, tokenizer), batch_size=32, shuffle=False)
    test_loader  = DataLoader(SimpleFusionDataset(test_df, tokenizer), batch_size=32, shuffle=False)

    # 2. 初始化模型
    # 这里以 HMCAN 为例，如果你用之前的 BERT+ResNet 记得切换类名
    model = HMCAN_Model(LOCAL_BERT_PATH).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    # 3. 设置保存相关的变量
    best_val_acc = 0.0
    # 设定模型保存的文件名（建议根据实验命名）
    MODEL_SAVE_PATH = "best_hmcan_model.pth" 

    # 4. 训练循环
    for epoch in range(5): 
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
            optimizer.zero_grad()
            outputs = model(batch["input_ids"].to(DEVICE), 
                            batch["attention_mask"].to(DEVICE), 
                            batch["pixel_values"].to(DEVICE))
            loss = criterion(outputs, batch["label"].to(DEVICE))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # 5. 验证阶段
        val_loss, val_acc, _, _ = evaluate_logic(model, val_loader, criterion, DEVICE)
        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # 6. 【核心保存逻辑】
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            # 将模型权重保存到磁盘
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"🌟 发现更好的模型！已保存至 {MODEL_SAVE_PATH}, Val Acc: {val_acc:.4f}")

    print("\n" + "="*30 + " 训练结束 " + "="*30)

    # 7. 加载保存的最佳权重进行最终测试
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"正在加载最佳模型权重: {MODEL_SAVE_PATH}")
        model.load_state_dict(torch.load(MODEL_SAVE_PATH))
        evaluate_on_test_set(model, test_loader, criterion, DEVICE)
    else:
        print("未找到保存的模型文件。")

In [9]:
from sklearn.metrics import classification_report, accuracy_score

@torch.no_grad()
def evaluate_logic(model, dataloader, criterion, device):
    """用于训练过程中的轻量化验证"""
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    for batch in dataloader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        imgs = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(ids, mask, imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, all_labels, all_preds

@torch.no_grad()
def evaluate_on_test_set(model, test_loader, criterion, device):
    """用于最后输出详细的测试报告"""
    _, acc, labels, preds = evaluate_logic(model, test_loader, criterion, device)
    
    print(f"\nFinal Results on Test Set:")
    print(f"Test Acc: {acc:.6f}")
    print("-" * 60)
    # 输出 precision, recall, f1 等
    print(classification_report(labels, preds, target_names=['real', 'fake'], digits=6))

In [10]:
train_model() # 取消注释运行

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1 [Train]: 100%|██████████| 171/171 [00:26<00:00,  6.40it/s]


Epoch 1: Train Loss: 0.6841 | Val Loss: 0.6602 | Val Acc: 0.7731
🌟 发现更好的模型！已保存至 best_hmcan_model.pth, Val Acc: 0.7731


Epoch 2 [Train]: 100%|██████████| 171/171 [00:24<00:00,  7.00it/s]


Epoch 2: Train Loss: 0.6438 | Val Loss: 0.5318 | Val Acc: 0.8039
🌟 发现更好的模型！已保存至 best_hmcan_model.pth, Val Acc: 0.8039


Epoch 3 [Train]: 100%|██████████| 171/171 [00:23<00:00,  7.17it/s]


Epoch 3: Train Loss: 0.6024 | Val Loss: 0.5013 | Val Acc: 0.8219
🌟 发现更好的模型！已保存至 best_hmcan_model.pth, Val Acc: 0.8219


Epoch 4 [Train]: 100%|██████████| 171/171 [00:24<00:00,  7.03it/s]


Epoch 4: Train Loss: 0.5792 | Val Loss: 0.4882 | Val Acc: 0.8236
🌟 发现更好的模型！已保存至 best_hmcan_model.pth, Val Acc: 0.8236


Epoch 5 [Train]: 100%|██████████| 171/171 [00:24<00:00,  7.04it/s]


Epoch 5: Train Loss: 0.5762 | Val Loss: 0.4837 | Val Acc: 0.8219

============================== 训练结束 ==============================
正在加载最佳模型权重: best_hmcan_model.pth

Final Results on Test Set:
Test Acc: 0.812660
------------------------------------------------------------
              precision    recall  f1-score   support

        real   0.772606  0.923688  0.841419       629
        fake   0.884892  0.683333  0.771160       540

    accuracy                       0.812660      1169
   macro avg   0.828749  0.803511  0.806290      1169
weighted avg   0.824475  0.812660  0.808964      1169



In [11]:
test_model = HMCAN_Model(LOCAL_BERT_PATH)
print(test_model)

/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


HMCAN_Model(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a